# Tutorial 04: Redes Neurais Convolucionais (CNNs) e Streaming I/O no FPGA

Neste tutorial, exploramos como acelerar **Redes Neurais Convolucionais (CNNs)** em FPGAs utilizando o `hls4ml`.

### Principais Tópicos Abordados:
1. **Arquiteturas Convolucionais no FPGA**: 1D vs 2D, convolução deslizante e pooling;
2. **Streaming I/O (`io_stream`)**: Por que o paralelismo total de fios (`io_parallel`) não escala para dados multidimensionais e como os barramentos AXI4-Stream e FIFOs (`hls::stream`) resolvem isso;
3. **Line Buffers e Shift Registers**: Como os dados de imagem/sequência fluem em hardware sem necessidade de armazenar o frame completo em memória externa;
4. **Dimensionamento de Ponto Fixo**: Análise da faixa dinâmica e por que 8 bits inteiros (`ap_fixed<16,8>`) são fundamentais para evitar overflow em somatórios convolucionais;
5. **C-Simulation Nativa**: Compilação C++ via `g++` e validação comparativa das Curvas ROC.

## 1. Importação de Bibliotecas e Configuração do Ambiente

In [ ]:
import os
import sys
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Configuração de caminhos
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from models import JetCNN1D
import hls4ml

print(f"Versão hls4ml : {hls4ml.__version__}")
print(f"Versão PyTorch: {torch.__version__}")

## 2. Carregando o Conjunto de Dados do CERN LHC
Formatamos o vetor de 16 variáveis físicas de jatos para a entrada convolucional de 1 canal: `(Batch, Canais=1, Comprimento=16)`.

In [ ]:
data_dir = os.path.join(ROOT_DIR, "data", "jet-tagging")
X_train = np.load(os.path.join(data_dir, "X_train.npy"))
y_train = np.load(os.path.join(data_dir, "y_train.npy"))
X_test = np.load(os.path.join(data_dir, "X_test.npy"))
y_test = np.load(os.path.join(data_dir, "y_test.npy"))
classes = np.load(os.path.join(data_dir, "classes.npy"), allow_pickle=True)

# Reshape para formato convolucional (Batch, Canais, Sequência)
X_train_cnn = np.ascontiguousarray(X_train[:, np.newaxis, :], dtype=np.float32)
X_test_cnn = np.ascontiguousarray(X_test[:, np.newaxis, :], dtype=np.float32)

print(f"Shape de Treino CNN : {X_train_cnn.shape}")
print(f"Shape de Teste CNN  : {X_test_cnn.shape}")
print(f"Classes de Partículas: {list(classes)}")

## 3. Modelo PyTorch (JetCNN1D)
Construímos uma rede convolucional contendo:
- `Conv1D(in_channels=1, out_channels=8, kernel_size=3, padding=1)`
- `ReLU`
- `MaxPool1D(kernel_size=2)` (reduzindo a dimensão temporal de 16 para 8)
- `Flatten`
- `Linear(64 -> 32)` -> `ReLU` -> `Linear(32 -> 5)` -> `Softmax(dim=-1)`

In [ ]:
device = torch.device("cpu")
model = JetCNN1D().to(device)

weights_path = os.path.join(ROOT_DIR, "models", "pytorch_weights_cnn.pt")
if os.path.exists(weights_path):
    print(f"Carregando pesos pré-treinados de {weights_path}...")
    model.load_state_dict(torch.load(weights_path, map_location=device))
else:
    print("Treinando modelo CNN...")
    n_train = 100000
    train_ds = TensorDataset(torch.from_numpy(X_train_cnn[:n_train]), torch.from_numpy(y_train[:n_train].astype(np.float32)))
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    for epoch in range(1, 6):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        print(f"Época {epoch}/5 finalizada.")
    torch.save(model.state_dict(), weights_path)

# Avaliação PyTorch Float32
model.eval()
eval_n = 10000
with torch.no_grad():
    y_pred_pytorch = model(torch.from_numpy(X_test_cnn[:eval_n])).cpu().numpy()
acc_pt = (np.argmax(y_pred_pytorch, axis=1) == np.argmax(y_test[:eval_n], axis=1)).mean()
print(f"Acurácia PyTorch (Float32): {acc_pt * 100:.2f}%")

## 4. Configuração de Hardware no hls4ml (`io_stream`)

### Por que `io_stream` é obrigatório para CNNs em FPGA?
- Em `io_parallel`, todos os pixels e neurônios precisariam de trilhas físicas dedicadas em paralelo simultaneamente, consumindo milhares de pinos de E/S (*IOBs*) e saturando o roteamento da FPGA;
- Em `io_stream`, os dados trafegam sequencialmente através de canais FIFO (**AXI4-Stream** com `hls::stream`). Um bloco de hardware especializado denominado **Line Buffer** armazena apenas as linhas/posições necessárias para formar a janela de convolução ($K \times K$).

### Dimensionamento de Precisão:
Configuramos `ap_fixed<16,8>` (8 bits inteiros, faixa de $[-128.0, +127.99]$) para acomodar os somatórios acumulados das ativações convolucionais sem sofrer *overflow wrap-around*.

In [ ]:
config = hls4ml.utils.config_from_pytorch_model(model, input_shape=(1, 16))
config["Model"]["Precision"]["default"] = "ap_fixed<16,8>"
config["Model"]["Strategy"] = "Latency"
config["Model"]["ReuseFactor"] = 1

output_dir = "hls4ml_prj_cnn"
hls_model = hls4ml.converters.convert_from_pytorch_model(
    model,
    input_shape=(1, 16),
    hls_config=config,
    output_dir=output_dir,
    part="xc7z020clg400-1",
    io_type="io_stream"
)
print("Projeto HLS gerado com sucesso!")

## 5. Compilação e C-Simulation Nativa (`g++`)

In [ ]:
t0 = time.time()
hls_model.compile()
print(f"Compilação g++ finalizada em {time.time() - t0:.2f}s.")

t1 = time.time()
y_pred_hls = hls_model.predict(X_test_cnn[:eval_n])
print(f"C-Simulation em {eval_n} eventos concluída em {time.time() - t1:.2f}s.")

acc_hls = (np.argmax(y_pred_hls, axis=1) == np.argmax(y_test[:eval_n], axis=1)).mean()
print(f"Acurácia PyTorch (Float32)    : {acc_pt * 100:.2f}%")
print(f"Acurácia hls4ml (ap_fixed<16,8>): {acc_hls * 100:.2f}%")
print(f"Diferença de Acurácia         : {(acc_pt - acc_hls) * 100:.2f}%")

## 6. Curvas ROC Comparativas (PyTorch vs hls4ml)

In [ ]:
plt.figure(figsize=(9, 7))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for i, label in enumerate(classes):
    fpr_pt, tpr_pt, _ = roc_curve(y_test[:eval_n, i], y_pred_pytorch[:, i])
    fpr_hls, tpr_hls, _ = roc_curve(y_test[:eval_n, i], y_pred_hls[:, i])
    auc_pt = auc(fpr_pt, tpr_pt)
    auc_hls = auc(fpr_hls, tpr_hls)
    plt.plot(fpr_pt, tpr_pt, color=colors[i], linestyle="--", alpha=0.7, label=f"{label} - PyTorch (AUC = {auc_pt:.3f})")
    plt.plot(fpr_hls, tpr_hls, color=colors[i], linestyle="-", lw=1.8, label=f"{label} - hls4ml (AUC = {auc_hls:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Baseline Aleatório")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("Taxa de Falso Positivo (FPR)", fontsize=12)
plt.ylabel("Taxa de Verdadeiro Positivo (TPR)", fontsize=12)
plt.title("Curvas ROC: PyTorch Float32 vs hls4ml ap_fixed<16,8> (CNN 1D)", fontsize=13, fontweight="bold")
plt.legend(loc="lower right", fontsize=9)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()